<a href="https://colab.research.google.com/github/vnandini867-ui/vnandini867/blob/main/CIT_COLLEGE_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U openai langchain-community langchain-huggingface langchain-text-splitters faiss-cpu beautifulsoup4 pypdf
import os
from openai import OpenAI
from google.colab import userdata

# LangChain Imports
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("Step 1: Ingesting data from multiple sources...")

# SOURCE A: Hardcoded Text / Paragraphs (The Detective Data)
raw_paragraphs =[
    'Witness A says: "Coorg Institute of Technology (CIT) is a private engineering college located in the scenic town of Ponnampet, Kodagu (Coorg), Karnataka."',
    'Witness B says: "The college offers engineering education."',
    'Witness C says: "The institution is located in Ponnampet."'
]
# Crucial Step: Convert raw Python strings into LangChain Document objects
text_documents = [Document(page_content=text) for text in raw_paragraphs]

# SOURCE B: Web Scraping (The College Data)
urls = [
    "https://www.citcoorg.edu.in/",
    "https://www.citcoorg.edu.in/about-cit/"
]
web_loader = WebBaseLoader(urls)
web_documents = web_loader.load()

# SOURCE C: PDF Documents
pdf_path = "/content/CIT_BROCHURE.pdf" # Instruct students to upload a PDF with this name
try:
    pdf_loader = PyPDFLoader(pdf_path)
    pdf_documents = pdf_loader.load()
    print("PDF loaded successfully.")
except Exception as e:
    print(f"Warning: PDF '{pdf_path}' not found in Colab. Skipping PDF ingestion.")
    pdf_documents = []

# MERGE ALL SOURCES
all_documents = text_documents + web_documents + pdf_documents
print(f"Total raw pages/documents loaded: {len(all_documents)}")

# Step 2: Data Chunking
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(all_documents)
print(f"Data chopped into {len(chunks)} searchable chunks.")

# Step 3: Embeddings & FAISS Vector Database
print("Building FAISS Vector Database...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db = FAISS.from_documents(chunks, embeddings)

# Step 4: Initialize LLM
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=userdata.get("nandu")
)

print("\n8🤖 Unified Universal Bot Online. Ask about the crime scene, college, or PDF.")
print("-" * 60)

# Step 5: The Chat Loop
while True:
    user_query = input("\nYou: ")
    if user_query.lower() == 'exit':
        break

    # Semantic Search across ALL sources simultaneously
    relevant_chunks = vector_db.similarity_search(user_query, k=3)
    retrieved_context = "\n\n".join([doc.page_content for doc in relevant_chunks])

    system_prompt = f"""
    You are an AI assistant powered by a multi-source knowledge base.
    Answer the user's question using ONLY the provided context.
    If the context does not contain the answer, say "I do not have enough information."
    Do NOT guess.

    CONTEXT:
    {retrieved_context}
    """

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query}
        ],
        temperature=0.0
    )

    print(f"AI: {response.choices[0].message.content}")

/tmp/ipykernel_16628/806523280.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader, PyPDFLoader


Step 1: Ingesting data from multiple sources...
PDF loaded successfully.
Total raw pages/documents loaded: 41
Data chopped into 174 searchable chunks.
Building FAISS Vector Database...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


8🤖 Unified Universal Bot Online. Ask about the crime scene, college, or PDF.
------------------------------------------------------------
AI: Coorg Institute of Technology (CIT) is a private engineering college located in the scenic town of Ponnampet, Kodagu (Coorg), Karnataka.
AI: I do not have enough information.
AI: **Dress Code (as stated in the provided context)**  

1. **General Attire**  
   - All students must wear only the prescribed decent dresses while attending the college on all days.  
   - Only neat and decent clothing is allowed.  
   - Boys and girls are to be dressed modestly and should have well‑groomed hair.  
   - Obscene dresses are not permitted.  
   - Repeated non‑adherence to this rule may result in severe action.  

2. **Lab/Workshop Specific**  
   - A specific colored apron must be worn.  
   - Black or brown leather shoes are required for workshops and laboratories.  

3. **Additional Notes**  
   - Students should not engage in noisy or unseemly behavio